In [8]:
import pandas as pd
import numpy as np
import hashlib 
import datetime

path = "scrapes\sensoryfair_adaptive_clothing.csv"
df=pd.read_csv(path)

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
C:\Users\21dan\AppData\Local\Temp\ipykernel_13720\2943117878.py:6: SyntaxWarning: invalid escape sequence '\s'
  path = "scrapes\sensoryfair_adaptive_clothing.csv"


In [9]:
def url_hash(u):
    if pd.isna(u): 
        return np.nan
    return hashlib.sha1(str(u).encode("utf-8")).hexdigest()[:12]

In [ ]:
df_out=df.copy()

df_out.insert(0,"product_id", range(1,len(df_out)+1))
df_out.insert(1,"url_hash", df_out["product_url"].apply(url_hash))
df_out.insert(2,"source", "sensoryfair.com")
df_out.insert(3,"scraped_at", pd.NaT)

# Standardize text fields for labeling convenience
# Create a combined text field to feed into LLM labeler
def combine_text(row):
    parts=[]
    for col in ["product_name","short_description","full_description"]:
        v=row.get(col, "")
        if pd.notna(v) and str(v).strip():
            parts.append(f"{col}: {str(v).strip()}")
    return "\n".join(parts)

df_out["text_for_labeling"]=df_out.apply(combine_text, axis=1)

# Add label columns (all empty/Unknown initially)
label_cols = {
    "seamless",                   # Yes/No/Unknown
    "fabric_softness",            # Low/Medium/High/Unknown
    "wool_present",               # Yes/No/Unknown
    "texture_level",              # Low/Medium/High/Unknown
    "pressure_level",             # Low/Medium/High/Unknown
    "adaptive_or_sensory_claim"   # Yes/No/Unknown
}

for col, default in label_cols.items():
    df_out[col]=default

# Evidence columns to make human review fast
for col in ["seamless_evidence","fabric_softness_evidence","wool_evidence",
            "texture_evidence","pressure_evidence","adaptive_claim_evidence"]:
    df_out[col]=""

# Workflow columns
df_out["labeler"]=""
df_out["label_status"]="unlabeled"   # unlabeled | auto_labeled | human_verified
df_out["review_notes"]=""
